# 3.4 Warning thresholds: calibration on 2023
**Purpose.** Give every warning score its final threshold, at both horizons, and freeze it for the 2024 run (paper Section 3.7). The 2024 run applies these thresholds and never chooses its own.

**Method.** The threshold of a score is the one with the highest CSI on the 2023 usable calm origins (`best_csi_threshold`), with a warning at every origin whose score is at least the threshold. This is done for every score in the same way: the linear models (no tuning), the XGBoost models (tuned on 2023 AP in notebook 3.1), the flow-ablation variant, the two baselines and SARIMA. **The 2023 POD, FAR and CSI below are optimistic**, because each threshold is chosen on 2023 itself; so is the 2023 AP of the XGBoost models, which chose their hyperparameters. All numbers here are validation observations; the comparisons of the study are made once, on 2024.

**Inputs and outputs.** The 2023 scores of notebooks 3.0 (linear, baselines), 3.1 (XGBoost), 3.2 (flow ablation) and 3.3 (SARIMA), from `data/interim/`. The thresholds, with the 2023 AP each score had, go to `models/thresholds-2023.json`. 2024 is not read.

## 1. The 2023 scores
**Method.** All score files must hold the same origins with the same targets; they are joined on year, horizon, district and slot.

In [1]:
import polars as pl

from madrid_congestion.config import HORIZONS, INTERIM_DATA_DIR, VALIDATION_YEAR
from madrid_congestion.evaluation import warning_table
from madrid_congestion.frozen import CONTRASTS, SCORES, check_design, load_frozen, save_frozen

SOURCES = ['linear', 'xgboost', 'ablation', 'sarima']  # the notebooks' score files, in the order of notebooks 3.0 to 3.3
KEYS = ["year", "minutes", "distrito", "slot_start"]
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)

frames = [pl.read_parquet(INTERIM_DATA_DIR / f"scores-{source}.parquet").filter(pl.col("year") == VALIDATION_YEAR) for source in SOURCES]
scores = frames[0]
for source, df in zip(SOURCES[1:], frames[1:], strict=True):
    joined = scores.join(df, on=KEYS, how="full", validate="1:1", coalesce=True)
    assert joined.height == scores.height == df.height, source  # the same origins
    assert (joined["target"] == joined["target_right"]).all(), source  # the same targets
    scores = joined.drop("target_right")
check_design(scores["distrito"].unique().sort())  # the code still builds the frozen design
present = [name for name in SCORES if name in scores.columns]  # in the order of the design's score list
assert sorted(present) == sorted(set(scores.columns) - {*KEYS, "target"}), "a score file holds an unknown score"
print(scores.group_by("minutes").agg(origins=pl.len(), positive=pl.col("target").sum()).sort("minutes"))
print(present, "| missing:", [name for name in SCORES if name not in present])

shape: (2, 3)
┌─────────┬─────────┬──────────┐
│ minutes ┆ origins ┆ positive │
│ ---     ┆ ---     ┆ ---      │
│ i32     ┆ u32     ┆ u32      │
╞═════════╪═════════╪══════════╡
│ 30      ┆ 482391  ┆ 8380     │
│ 60      ┆ 461884  ┆ 16275    │
└─────────┴─────────┴──────────┘
['proximity', 'calendar', 'linear_occupancy', 'linear_flow', 'linear_direct', 'xgboost_occupancy', 'xgboost_flow', 'xgboost_direct', 'xgboost_direct_no_flow', 'sarima_occupancy', 'sarima_flow'] | missing: []


## 2. Thresholds and warning measures
**Method.** For each horizon and score: AP with the positive-origin base rate, the highest-CSI warning threshold (a cut on the warning score, not the high-occupancy threshold of the label), and POD, FAR and CSI at it, with the origin counts behind them (hits: warned positive origins; misses: positive origins without a warning; false alarms: warnings at negative origins).

In [2]:
calibration, frozen = [], {}
for minutes in HORIZONS:
    df = scores.filter(pl.col("minutes") == minutes)
    table = warning_table(df["target"], {name: df[name] for name in present})
    frozen[str(minutes)] = {row["score"]: {"warning_threshold": row["warning_threshold"], "ap_2023": row["ap"]} for row in table.iter_rows(named=True)}
    calibration.append(table.select(pl.lit(minutes).alias("minutes"), pl.lit(df["target"].mean()).alias("base_rate"), pl.all()))
save_frozen("thresholds-2023.json", frozen)
assert load_frozen("thresholds-2023.json") == frozen  # exact floats survive the file
pl.concat(calibration).with_columns(pl.col("base_rate", "ap", "pod", "far", "csi").round(4), pl.col("warning_threshold").round(6))

minutes,base_rate,score,ap,warning_threshold,pod,far,csi,hits,misses,false_alarms
i32,f64,str,f64,f64,f64,f64,f64,i64,i64,i64
30,0.0174,"""proximity""",0.2483,0.908243,0.4765,0.7287,0.209,3993,4387,10723
30,0.0174,"""calendar""",0.3521,0.191667,0.4611,0.6378,0.2545,3864,4516,6803
30,0.0174,"""linear_occupancy""",0.4149,0.959307,0.5463,0.6261,0.2853,4578,3802,7665
30,0.0174,"""linear_flow""",0.3576,1.028972,0.4143,0.6356,0.2405,3472,4908,6056
30,0.0174,"""linear_direct""",0.5172,0.218178,0.514,0.543,0.3191,4307,4073,5118
30,0.0174,"""xgboost_occupancy""",0.6961,1.003616,0.6632,0.3805,0.4713,5558,2822,3414
30,0.0174,"""xgboost_flow""",0.5331,1.022491,0.5193,0.4941,0.3446,4352,4028,4250
30,0.0174,"""xgboost_direct""",0.7202,0.283188,0.6896,0.3891,0.4791,5779,2601,3681
30,0.0174,"""xgboost_direct_no_flow""",0.716,0.299582,0.6766,0.3754,0.481,5670,2710,3408


## 3. The comparisons, as 2023 observations
**Method.** The differences in 2023 AP for the comparisons the study makes on 2024 (paper Section 5.1): the primary contrast, direct minus occupancy (XGBoost), and the secondary ones, occupancy minus flow (XGBoost) and, when notebook 3.2 has run, the flow ablation, direct with flow minus direct without flow. These are point values on the year that tuned the models, with no interval; they decide nothing.

In [3]:
pl.DataFrame([
    {"minutes": minutes, "comparison": name, "ap_first": frozen[str(minutes)][a]["ap_2023"], "ap_second": frozen[str(minutes)][b]["ap_2023"],
     "difference": frozen[str(minutes)][a]["ap_2023"] - frozen[str(minutes)][b]["ap_2023"]}
    for minutes in HORIZONS for name, (a, b) in CONTRASTS.items() if b in present  # CONTRASTS: frozen.py
]).with_columns(pl.col("ap_first", "ap_second", "difference").round(4))

minutes,comparison,ap_first,ap_second,difference
i64,str,f64,f64,f64
30,"""primary: direct - occupancy""",0.7202,0.6961,0.0241
30,"""secondary: occupancy - flow""",0.6961,0.5331,0.163
30,"""secondary: with flow - without…",0.7202,0.716,0.0042
60,"""primary: direct - occupancy""",0.7822,0.7578,0.0244
60,"""secondary: occupancy - flow""",0.7578,0.6332,0.1246
60,"""secondary: with flow - without…",0.7822,0.7761,0.0061


**Result.** To be written after the run.